# Python Basics: Interactive Review

A practical refresher for students preparing for AI, search, and data-science coursework.

**Audience:** students who have seen basic programming. 
**Prerequisites:** variables, conditions, loops, and functions. 
**Estimated time:** 90-120 minutes.

This notebook uses the Python standard library only. Try each prompt before running the worked example below it. Run cells from top to bottom; restarting the kernel and choosing Run All should complete without errors.

By the end, you should be able to choose useful containers, write reusable functions, handle expected errors, read and write a small text file safely, and test your assumptions.

In [2]:
import sys

print("Python version:", sys.version.split()[0])
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."

Python version: 3.13.7


## 1. Quick diagnostic

Predict before running later examples. Note any question you cannot yet answer; these are the concepts to revisit.

1. What is the difference between `==` and `is`?
2. What does `dict(zip(["x", "y"], [1]))` contain?
3. If a function prints a value but does not return one, what does its caller receive?
4. Why can an outer-list copy still share nested data?
5. Which operations make a list act as a stack and a deque act as a queue?

1. What is the difference between `==` and `is`?

<details>

<summary><b>Click to show answer: Difference between <code>==</code> and <code>is</code></b></summary>

<br>

The difference between `==` and `is` comes down to **equality vs. identity**:

* **`==` (Equality):** Compares the **values** or contents of two objects (`__eq__()`). It checks: *"Do these two objects have equivalent data?"*
* **`is` (Identity):** Compares the **memory addresses** of two objects (`id()`). It checks: *"Are these two variables pointing to the exact same object in memory?"*

---

### Key Comparison

| Operator | Checked Condition | Underlying Mechanism | Typical Use Case |
| :--- | :--- | :--- | :--- |
| **`==`** | `a` and `b` have equivalent content | `a.__eq__(b)` | Numeric comparisons, string matching, data equality |
| **`is`** | `id(a) == id(b)` | Memory address comparison | Singletons (`None`, `True`, `False`), sentinel values |

---

### Demonstrating the Difference

#### 1. Equivalent Content, Distinct Memory Locations
```python
list_a = [1, 2, 3]
list_b = [1, 2, 3]

print(list_a == list_b)  # True  -> contents are identical
print(list_a is list_b)  # False -> separate heap allocations
print(id(list_a) == id(list_b))  # False
```

#### 2. Aliasing (Shared Memory Reference)
```python
list_c = list_a  # Pointer assignment, no new allocation

print(list_a == list_c)  # True  -> same contents
print(list_a is list_c)  # True  -> pointing to identical memory address
```

### AI & Scientific Computing Caveats

#### 1. Singletons and Default Arguments
In Python libraries and AI frameworks (e.g., PyTorch, Hugging Face transformers), optional arguments default to `None`. Always check for `None` using `is`, never `==`:

```python
# Correct: fast pointer check, immune to custom __eq__ implementations
if learning_rate is None:
    learning_rate = 1e-3

# Avoid: slower, invokes __eq__ which may be overridden
if learning_rate == None:
    ...
```

#### 2. The Small Integer & String Interning Trap
CPython caches/interns small integers (typically range `[-5, 256]`) and short strings for optimization. Relying on `is` for integers leads to subtle bugs:

```python
a = 256
b = 256
print(a is b)  # True (due to CPython small integer caching)

x = 1000
y = 1000
print(x is y)  # False (allocated as distinct objects)
print(x == y)  # True  (always compare numeric values with ==)
```

</details>

2. What does `dict(zip(["x", "y"], [1]))` contain?

<details>
<summary><b>Click to show answer: <code>dict(zip(["x", "y"], [1]))</code></b></summary>

<br>

**Result:**

```python
{'x': 1}
```

---

### Why This Happens

`zip()` pairs elements from each iterable index-by-index and stops as soon as the **shortest** iterable is exhausted:

1. `["x", "y"]` has length 2.
2. `[1]` has length 1.
3. `zip(["x", "y"], [1])` produces the single tuple `('x', 1)` and terminates immediately, silently dropping `"y"`.
4. `dict(...)` evaluates that single key-value pair, producing `{'x': 1}`.

---

### AI & Data Pipeline Caveats

In feature mapping, hyperparameter construction, or tokenizer alignment, silent truncation caused by uneven list lengths can introduce silent bugs. 

To handle mismatched lengths explicitly:

* **Strict mode (Python 3.10+):** Raise an exception if iterables are not equal in length:
  ```python
  dict(zip(["x", "y"], [1], strict=True))
  # Raises ValueError: zip() argument 2 is shorter than argument 1
  ```

* **Padding with defaults:** Preserve all keys by using `itertools.zip_longest`:
  ```python
  from itertools import zip_longest

  dict(zip_longest(["x", "y"], [1], fillvalue=None))
  # Returns: {'x': 1, 'y': None}
  ```

</details>

3. If a function prints a value but does not return one, what does its caller receive?

<details>
<summary><b>Click to show answer: What does a caller receive if a function prints but has no return?</b></summary>

<br>

The caller receives **`None`**.

In Python, every function returns a value. If execution reaches the end of a function body without encountering an explicit `return` statement (or hits an empty `return`), Python implicitly returns the `None` singleton object.

---

### Demonstrating the Behavior

```python
def log_loss(loss_val):
    print(f"Current Loss: {loss_val}")  # Side effect (writes to stdout)
    # No return statement here

result = log_loss(0.042)
# Output on console: Current Loss: 0.042

print(result)          # None
print(type(result))    # <class 'NoneType'>
print(result is None)  # True
```

---

### Common Bug in AI & Data Pipelines

A frequent pitfall in ML preprocessing pipelines occurs when a function mutates data or logs diagnostics but forgets to return the modified object:

```python
def normalize_data(data):
    data = (data - data.mean()) / data.std()
    # Forgot: return data

# Attempting to assign downstream leads to NoneType failures:
X_norm = normalize_data(X)
model.fit(X_norm)  
# Raises: AttributeError / TypeError because X_norm is None, not a tensor/array
```

#### Built-in In-Place Methods Follow the Same Rule:
Python standard methods that mutate objects in place return `None` by design to avoid accidental reassignment:
* `my_list.sort()` sorts in-place and returns **`None`** (unlike `sorted(my_list)`, which returns a new list).
* `my_list.append(x)` mutates the list and returns **`None`**.

</details>

4. Why can an outer-list copy still share nested data?

<details>
<summary><b>Click to show answer: Why an outer-list copy still shares nested data</b></summary>

<br>

### The Core Reason: Shallow Copying vs. Pointers

In Python, variables and container elements do not store objects directly; **they store memory references (pointers)**.

When you copy an outer list using standard techniques (`.copy()`, `list()`, slicing `[:]`, or list comprehensions), Python creates a **shallow copy**:
1. A **new outer container** is allocated with its own unique identity and memory address.
2. The **pointers** to the elements inside the original list are copied verbatim into the new container.
3. The nested objects themselves are **not** cloned or duplicated.

```text
Memory Reference Layout:

original ----> [ ptr_0 | ptr_1 ]
                  |        |
                  |        +--------> [3, 4] (shared heap object)
                  v
               [1, 2] (shared heap object)
                  ^
                  |
shallow  ----> [ ptr_0 | ptr_1 ]
```

Because both `original[0]` and `shallow[0]` store the identical memory address, performing an in-place mutation on that object via one list alters the data seen by the other.

---

### Comprehensive Breakdown: All 4 Cases

#### Case 1: In-Place Mutation of a Nested Mutable Object (Shared Side Effect)
Mutating an inner list (via `.append()`, `.extend()`, `.pop()`, etc.) modifies the object in place on the heap without changing any pointers in either outer list.

```python
original = [[1, 2], [3, 4]]
shallow = original.copy()

print(original is shallow)        # False (different outer lists)
print(original[0] is shallow[0])  # True  (exact same inner list)

# In-place mutation via the shallow copy
shallow[0].append(999)

print(original)  # [[1, 2, 999], [3, 4]] -> MUTATED!
print(shallow)   # [[1, 2, 999], [3, 4]]
```

---

#### Case 2: Rebinding a Top-Level Element (Independent)
Reassigning an index of the outer copied list overwrites only that slot's pointer in the new container. The original list continues pointing to the original object.

```python
original = [[1, 2], [3, 4]]
shallow = original.copy()

# Rebind index 0 of the outer copy to point to a new object
shallow[0] = ["a", "b"]

print(original[0] is shallow[0])  # False (pointers now differ)
print(original)  # [[1, 2], [3, 4]] -> Unaffected!
print(shallow)   # [['a', 'b'], [3, 4]]
```

---

#### Case 3: Rebinding *Inside* a Nested Object (Shared Side Effect)
Reassigning a value inside an inner collection (e.g., `shallow[0][1] = 99`) mutates the inner object's internal slot. Because that inner object is shared, both outer lists reflect the change.

```python
original = [[10, 20], [30, 40]]
shallow = original[:]

shallow[0][1] = 999  # Mutates the shared inner list at index 0

print(original)  # [[10, 999], [30, 40]] -> Reflected in original!
print(shallow)   # [[10, 999], [30, 40]]
```

---

#### Case 4: Nested Immutable Objects (Safe by Immutability)
If the nested items are immutable (e.g., numbers, strings, tuples), the references are still shared, but the objects cannot be mutated in place. Any update requires creating a new object or rebinding, preventing unintended side effects.

```python
original = [(1, 2), (3, 4)]
shallow = list(original)

print(original[0] is shallow[0])  # True (shared reference)

# Tuples cannot be appended or modified in place.
# Reassigning forces a new pointer assignment on the outer list:
shallow[0] = (1, 2, 999)

print(original)  # [(1, 2), (3, 4)] -> Untouched!
print(shallow)   # [(1, 2, 999), (3, 4)]
```

> **Subtle Edge Case:** If an immutable tuple holds an inner mutable list (e.g., `[([1, 2], 3)]`), calling `shallow[0][0].append(99)` **will** still mutate `original` because the underlying list remains mutable and shared.

---

### Common Operations That Create Shallow Copies

| Technique | Syntax |
| :--- | :--- |
| **Method Call** | `new_list = original.copy()` |
| **Full Slicing** | `new_list = original[:]` |
| **Type Constructor** | `new_list = list(original)` |
| **List Comprehension** | `new_list = [x for x in original]` |
| **Standard Library** | `import copy; new_list = copy.copy(original)` |
| **List Multiplication** | `nested = [[0]] * 3` *(replicates the same inner reference 3 times!)* |

---

### The Fix: Deep Copying

To recursively allocate brand-new independent objects at every level of nesting, use `copy.deepcopy()`:

```python
import copy

original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)

print(original is deep)        # False
print(original[0] is deep[0])  # False (new inner list created)

deep[0].append(999)

print(original)  # [[1, 2], [3, 4]] -> Intact!
print(deep)      # [[1, 2, 999], [3, 4]]
```

</details>

5. Which operations make a list act as a stack and a deque act as a queue?



## 2. Names, values, and types

A variable is a name bound to an object. Assignment does not automatically copy that object. Use `==` to compare values and `is` for identity checks such as `value is None`. Prefer `isinstance` for most type checks.

In [3]:
count = 7
temperature = 21.5
label = "sample"
available = True
missing = None

print(type(count).__name__, type(temperature).__name__)
print(bool(0), bool([]), missing is None)
assert isinstance(count, int) and isinstance(temperature, float)
assert count == 7 and label == "sample"

int float
False False True


## 3. Collections and comprehensions

Lists are ordered and mutable; tuples are ordered and immutable; sets contain unique hashable values; dictionaries map hashable keys to values. Choose a container based on the operations your code needs.

**Predict:** which values remain after converting the list to a set? What happens when the lists passed to `zip` have different lengths?

In [ ]:
scores = [8, 5, 8, 10]
point = (2, 3)
unique_scores = set(scores)
score_by_name = {"Ada": 8, "Lin": 10}

doubled = [score * 2 for score in scores]
high_scores = {score for score in scores if score >= 8}
names_by_score = {name: score for name, score in score_by_name.items()}

print(unique_scores, doubled, high_scores, names_by_score)
print(list(zip(["a", "b"], [1])))
assert doubled == [16, 10, 16, 20]
assert high_scores == {8, 10}
assert len(list(zip(["a", "b"], [1]))) == 1

For data that must align exactly, use `zip(..., strict=True)` (Python 3.10+) so mismatched lengths raise `ValueError` rather than silently truncating.

In [ ]:
aligned = dict(zip(["a", "b"], [10, 20], strict=True))
assert aligned == {"a": 10, "b": 20}
print(aligned)

## 4. Iteration and control flow

`for` works with any iterable. `enumerate` pairs values with an index; `range` produces integer sequences. A loop's `else` clause runs when the loop finishes without `break`; `continue` does not prevent it.

In [ ]:
for index, score in enumerate(scores, start=1):
    if score < 6:
        continue
    print(index, score)
else:
    print("Finished without break")

numbers = [10, 20, 30]
iterator = iter(numbers)
assert next(iterator) == 10
assert list(iterator) == [20, 30]

## 5. Functions and contracts

A function should return a result when callers need to use it. A docstring describes its purpose; type hints communicate expected values but are not runtime enforcement.

**Try first:** write a function that returns the average of a non-empty list. Decide what the function should do for an empty list.

In [ ]:
def average(values: list[float]) -> float:
    """Return the arithmetic mean of a non-empty list of numbers."""
    if not values:
        raise ValueError("values must not be empty")
    return sum(values) / len(values)

assert average([2, 4, 6]) == 4
print(average([2, 4, 6]))

In [ ]:
try:
    average([])
except ValueError as error:
    print("Handled invalid input:", error)
else:
    raise AssertionError("Expected an empty-list error")

### Parameters, defaults, and callbacks

Positional-only parameters use `/`; keyword-only parameters follow `*`. Avoid mutable defaults because the same object is reused across calls. Use `None` when a fresh mutable object is needed.

In [ ]:
def describe(value, /, *, uppercase=False):
    text = str(value)
    return text.upper() if uppercase else text

def append_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

assert describe("ai", uppercase=True) == "AI"
assert append_item(1) == [1]
assert append_item(2) == [2]

In [ ]:
from collections.abc import Callable

def rank_values(values: list[int], score: Callable[[int], int]) -> list[int]:
    """Return values sorted by the supplied scoring function."""
    return sorted(values, key=score)

values = [5, 2, 9, 1]
assert rank_values(values, lambda value: value) == [1, 2, 5, 9]
assert rank_values(values, lambda value: -value) == [9, 5, 2, 1]
assert values == [5, 2, 9, 1]

`*args` collects extra positional arguments into a tuple; `**kwargs` collects extra named arguments into a dictionary. Use them when variable arguments are part of a clear interface, not as a substitute for explicit parameters.

In [ ]:
def show_call(first, *args, **kwargs):
    return first, args, kwargs

result = show_call(1, 2, 3, mode="fast")
assert result == (1, (2, 3), {"mode": "fast"})
print(result)

## 6. Modules and imports

A module is an importable unit, commonly a `.py` file. Import named functionality from the standard library rather than copying code or using wildcard imports. A package is a way to organize related modules; regular packages commonly include `__init__.py`, while namespace packages may not.

Python searches locations in `sys.path`. The initial search location depends on how Python was launched, so run project scripts from the intended project environment and directory.

In [ ]:
import math
from statistics import median

assert math.isclose(math.sqrt(9), 3.0)
assert median([1, 9, 3]) == 3
print(math.pi, median([1, 9, 3]))

## 7. Exceptions, debugging, and tests

Read the exception type, message, and traceback location. Catch only errors you can handle meaningfully. Assertions are useful for development checks, but they are not a replacement for validating external input in production code.

In [ ]:
def item_at(values: list[int], index: int) -> int:
    if not isinstance(index, int) or isinstance(index, bool):
        raise TypeError("index must be an integer")
    if not 0 <= index < len(values):
        raise IndexError("index is outside the list")
    return values[index]

assert item_at([4, 8], 1) == 8
for bad_index, expected_error in [(True, TypeError), (-1, IndexError), (2, IndexError)]:
    try:
        item_at([4, 8], bad_index)
    except expected_error:
        pass
    else:
        raise AssertionError(f"Expected {expected_error.__name__}")

## 8. File handling

Use a context manager so the file is closed even if an error occurs. Specify an encoding for text files. This example uses a temporary directory, so it leaves no project files behind.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "scores.txt"
    with path.open("w", encoding="utf-8") as file:
        file.write("Ada,8\nLin,10\n")
    with path.open(encoding="utf-8") as file:
        rows = file.read().splitlines()
    assert rows == ["Ada,8", "Lin,10"]
    print(rows)

## 9. A small OOP example

A dataclass is useful for records with named fields. A frozen dataclass prevents field reassignment. Keep the representation focused on the data and behavior the problem actually needs.

In [ ]:
from dataclasses import dataclass

Coordinate = tuple[int, int]

@dataclass(frozen=True)
class SearchNode:
    state: Coordinate
    parent_state: Coordinate | None = None
    path_cost: int = 0

node = SearchNode((2, 3), (2, 2), 4)
assert node.state == (2, 3)
assert hash(node) == hash(SearchNode((2, 3), (2, 2), 4))
print(node)

## 10. Reproducible randomness

When an experiment uses randomness, record the seed and other relevant settings. A local random-number generator avoids changing the process-wide random state. A seed makes this small example repeatable; it does not by itself guarantee identical results across every library, device, or software version.

In [ ]:
import random

first_run = random.Random(42).sample(range(100), k=5)
second_run = random.Random(42).sample(range(100), k=5)
assert first_run == second_run
print(first_run)

## 11. Integrated practice

Before running the worked solution, implement `valid_coordinate` yourself. It should accept a two-integer tuple, reject booleans, and return whether the coordinate lies inside `(rows, columns)`. Add one test for a boundary coordinate and one test for malformed input.

In [ ]:
def valid_coordinate(state: object, *, size: tuple[int, int]) -> bool:
    """Return whether state is an in-bounds pair of non-Boolean integers."""
    if not isinstance(state, tuple) or len(state) != 2:
        return False
    row, column = state
    if any(not isinstance(value, int) or isinstance(value, bool) for value in state):
        return False
    rows, columns = size
    return 0 <= row < rows and 0 <= column < columns

assert valid_coordinate((0, 0), size=(3, 5)) is True
assert valid_coordinate((2, 4), size=(3, 5)) is True
assert valid_coordinate((3, 0), size=(3, 5)) is False
assert valid_coordinate((True, 0), size=(3, 5)) is False
assert valid_coordinate([0, 0], size=(3, 5)) is False
print("PASS: coordinate validation")

## Readiness checklist

You are ready to move on when you can explain or demonstrate each item without copying the example:

- Choose between a list, tuple, set, and dictionary for a stated task.
- Explain aliasing, shallow copies, and mutable default arguments.
- Use a comprehension, `enumerate`, and `zip` appropriately.
- Write a function with a clear return contract and handle invalid inputs.
- Use imports, a context manager for files, and a focused exception handler.
- Write assertions for normal cases, boundaries, and malformed inputs.
- Explain how a local random seed helps reproduce an experiment.

## Next: Python for data and AI

This notebook intentionally does not teach numerical or tabular data analysis. Before ML coursework, continue with NumPy arrays (`shape`, `dtype`, indexing, and vectorized operations), pandas DataFrames, missing-data handling, basic visualization, and train/test separation. Keep model and experiment code reproducible by recording data versions, parameters, and random seeds.

For search and planning, the next step is to reuse these containers, function contracts, and tests to implement a problem interface and then study search algorithms.